# Machine Doctor — Deep Learning Add-on
## Step 3: Build the Neural Network

A **1D Convolutional Neural Network (CNN)** -- good at spotting shapes/patterns in a signal, which is exactly what a vibration waveform is. This is a small, well-established architecture style for this exact task (similar in spirit to "WDCNN", a well-known CWRU-benchmark CNN that uses a wide first-layer kernel to capture low-frequency information even from a short window).

This step ONLY builds and sanity-checks the model shape -- no training yet. That's Step 4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import torch
import torch.nn as nn

data = np.load('/content/drive/MyDrive/machine_doctor_dl/processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test, y_test = data['X_test'], data['y_test']
class_names = list(data['class_names'])

print(f"X_train: {X_train.shape}, classes: {class_names}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class VibrationCNN(nn.Module):
    """
    Kid explanation: this is the network's "brain structure." Each
    Conv1d layer is like a little detector sliding along the signal,
    learning to recognize a specific small shape (a spike, a ripple, a
    flat patch). Stacking several layers lets it build up from simple
    shapes to complex fault signatures, the same way your visual system
    builds up from edges to whole objects.

    The FIRST layer uses a wide kernel (64) on purpose -- a common trick
    in vibration-CNN papers, since a wide kernel can "see" enough of the
    signal at once to capture low-frequency patterns even though our
    window is short.
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=64, stride=2, padding=32),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=16, stride=2, padding=8),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = VibrationCNN(num_classes=len(class_names)).to(device)
print(model)

In [ ]:
# Sanity check: feed ONE real batch through the UNTRAINED model and
# confirm the output shape is exactly right (batch_size, num_classes)
# before we invest any time in training. Catching a shape bug now is
# free; catching it after a 10-minute training run is not.
sample_batch = torch.tensor(X_train[:8]).unsqueeze(1).to(device)  # add channel dim: (batch, 1, 1024)
print("Input shape:", sample_batch.shape)

model.eval()
with torch.no_grad():
    output = model(sample_batch)
print("Output shape:", output.shape, " (expected: (8,", len(class_names), "))")

assert output.shape == (8, len(class_names)), "Shape mismatch -- fix before training!"
print("\nShape check passed. Model architecture is valid and ready for Step 4 (training).")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params:,} (small -- this is why it trains fast even on a free GPU)")